# Lab 1 — Retrieval & Document-Processing Agents

**What this lab is.** We build the first two "specialist agents": the **Retrieval Agent**
(which searches the approved Knowledge Base) and the **Document-Processing Agent** (which
tidies the found text into a clear, structured answer). "Agent" here just means an AI model
given a specific job and strict rules.

**Why we do it.** CareConnect never answers from the model's general knowledge — only from
approved documents. The Retrieval Agent is what enforces that: it can *only* return what the
Knowledge Base gives it. The Document-Processing Agent then makes that raw text readable
without adding anything new.

**Why it's needed here.** Splitting the work into small, single-purpose agents keeps each one
easy to reason about and to keep safe. A retrieval agent that can only quote approved
documents is far safer than one big model free-styling answers.

**How it helps the project.** These two agents become the "knowledge" half of the final
assistant. The Supervisor (lab-05) will call them whenever a patient asks something that
should be answered from documents.

**The use case.** "How do I prepare for my endoscopy?" → the Retrieval Agent finds the
approved leaflet; the Document-Processing Agent turns it into a clean checklist — with the
source cited, and nothing invented.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [2]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Load helpers and confirm the Knowledge Base

**What:** import our helpers and the two agent builders, and print the Knowledge Base ID.

**Why:** this confirms lab-00 succeeded and the agents will point at the right Knowledge
Base. If the ID doesn't print, lab-00 wasn't completed.

In [3]:
# WHAT THIS CELL DOES (plain English):
# - Loads our helpers and the two functions that build the agents.
# - Prints the Knowledge Base ID (read from Parameter Store) so you can confirm lab-00 ran.
import lab_helpers.utils as u
from lab_helpers.careconnect_agents import build_retrieval_agent, build_docproc_agent
print("KB in use:", u.get_ssm_parameter(f"{u.SSM_PREFIX}/kb_id"))

KB in use: ZI0R8MDWXM


### Step 2 — The Retrieval Agent

**What:** create the Retrieval Agent and ask it a sample question.

**Why:** this agent's only tool is "search the approved documents". It is instructed to use
*only* what it finds and to say clearly when the documents don't cover something — which is
exactly what you want in a hospital assistant.

In [4]:
# WHAT THIS CELL DOES (plain English):
# - Builds the Retrieval Agent (an AI model whose only tool is searching the Knowledge Base).
# - Asks it a sample question and prints the answer, which will be based only on approved docs.
# - Notice in the output it honestly says when the docs don't contain the answer.
retrieval_agent = build_retrieval_agent()
resp = retrieval_agent("How should a patient prepare for a colonoscopy?")
print(resp)


Tool #1: search_docs
I understand you're asking about colonoscopy preparation, but the approved Riverside Health documents I have access to don't contain specific colonoscopy preparation instructions. The documents I retrieved cover preparation for upper endoscopy, CT scans, general appointments, and fasting blood tests, but none specifically address colonoscopy prep.

To get the correct colonoscopy preparation instructions, I recommend:

1. **Contact your care team directly** - They will provide you with the specific colonoscopy preparation instructions that are appropriate for your individual health situation.

2. **Check your appointment materials** - If you've already scheduled your colonoscopy, look for any pre-appointment instructions or paperwork you may have received from Riverside Health.

3. **Ask about the specific prep regimen** - Colonoscopy preparation typically involves a clear liquid diet and bowel prep solution, but the exact instructions vary based on the type of pre

### Step 3 — The Document-Processing Agent

**What:** create the agent that takes the retrieved text and reformats it into a clear,
structured answer (e.g. a numbered checklist).

**Why:** raw retrieved text can be messy. This agent makes it readable **without adding any
new medical content** — it only reorganizes what the documents already say.

In [5]:
# WHAT THIS CELL DOES (plain English):
# - Builds the Document-Processing Agent.
# - Feeds it the text the Retrieval Agent found (as DATA, not as instructions) and asks it to
#   produce a clean numbered checklist using ONLY that text.
docproc_agent = build_docproc_agent()

# Feed the retrieved passages from the retrieval agent as DATA.
evidence = str(resp)
structured = docproc_agent(
    "Convert the following approved evidence into a clear numbered checklist. "
    "Use ONLY what is in the evidence.\n\n<evidence>\n" + evidence + "\n</evidence>")
print(structured)

## Colonoscopy Preparation Guidance from Riverside Health

Based on the approved Riverside Health documents retrieved, here is the guidance provided regarding colonoscopy preparation:

1. **Contact your care team directly** - They will provide you with the specific colonoscopy preparation instructions that are appropriate for your individual health situation.

2. **Check your appointment materials** - If you've already scheduled your colonoscopy, look for any pre-appointment instructions or paperwork you may have received from Riverside Health.

3. **Ask about the specific prep regimen** - While general colonoscopy preparation typically involves a clear liquid diet and bowel prep solution, the exact instructions vary based on the type of prep being used and your personal health factors.

**Important Note:** The retrieved Riverside Health documents specifically state that they do not contain colonoscopy preparation instructions. The above points represent recommended next steps from the

### Step 4 — Safety check: it must not invent content

**What:** feed the agent an "empty" result (no approved info found) and confirm it refuses to
make anything up.

**Why:** this is the important negative test. A safe assistant, when it has no approved
information, must *say so* — not fill the gap with plausible-sounding but unverified advice.

In [6]:
# WHAT THIS CELL DOES (plain English):
# - Gives the agent an input that says "no approved information was found".
# - A safe agent should simply report that nothing was found and NOT invent instructions.
# - The output confirms it behaves correctly.
empty = docproc_agent(
    "Structure this evidence:\n<evidence>\nNo approved Riverside Health "
    "information was found for this question.\n</evidence>")
print(empty)  # should decline to invent instructions

## Information Status

The approved Riverside Health documents retrieved do not contain any information addressing this question. No approved Riverside Health information was found for this topic.## Information Status

The approved Riverside Health documents retrieved do not contain any information addressing this question. No approved Riverside Health information was found for this topic.



## Lab 1 complete ✅

Retrieval and Document-Processing agents work against the `-sdk` KB.